# Импорт библиотек

In [6]:
import sys
import joblib
import pandas as pd

In [5]:
sys.path.append('../src') 

from models import lgbm_quantile

OSError: dlopen(/Users/dataalph/Desktop/da_venv/lib/python3.10/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <8FC36893-94B8-343C-9D9F-4CCBFE81B89B> /Users/dataalph/Desktop/da_venv/lib/python3.10/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volume

# Подгрузка датафреймов

In [3]:
df_train = pd.read_pickle('../data/df_train.pkl')
df_test = pd.read_pickle('../data/df_test.pkl')

# Модель

In [ ]:
wmae_quantile, quantile_model = lgbm_quantile(df_train, alpha=0.5)

# Сохраняем модель 
model_package = {
    'model': quantile_model,
    'feature_names': df_train.drop(['id', 'w', 'target'], axis=1, errors='ignore').columns.tolist(),
    'alpha': 0.5
}

models_path = '../models/trained_model_lgbm_quantile.joblib'
joblib.dump(model_package, models_path)
print("Model saved as 'trained_model_lgbm_quantile.joblib'")

# Загружаем и делаем предсказания
loaded_model_package = joblib.load(models_path)
model = loaded_model_package['model']
feature_names = loaded_model_package['feature_names']

print(f"Model loaded: {len(feature_names)} features")

X_test = df_test[feature_names]
predictions = model.predict(X_test)

# Создаем submission файл 
submission = pd.DataFrame({
    'id': df_test['id'],
    'target': predictions
})

submission_path = '../models/trained_model_lgbm_quantile.csv'
submission.set_index('id').to_csv(submission_path, decimal='.', sep=',')
print(f"Submission file saved as '{submission_path}'")